In [1]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

import yfinance as yf
import matplotlib.pyplot as plt

In [2]:


SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

TICKER = "BTC-USD"
START = "2018-01-01"
END = None

TRAIN_FRAC = 0.8

# Trading / reward parameters
FEE = 0.0005         # transaction cost per unit position change
KAPPA = 0.01          # risk penalty weight

# PPO hyperparameters (good starting point for toy project)
num_envs = 8
n_steps = 128
total_updates = 2000

gamma = 0.99
gae_lambda = 0.95

lr = 3e-4
vf_coef = 0.5
ent_coef = 0.001
max_grad_norm = 0.5

clip_eps = 0.2
ppo_epochs = 10
minibatch_size = 256
target_kl = 0.1 # experiment: 0.05 oder 0.15

In [3]:
def load_ohlcv(ticker, start, end=None, interval="1d"):
    df = yf.download(
        ticker,
        start=start,
        end=end,
        interval=interval,
        auto_adjust=True,
        progress=False
    )

    df = df.dropna()

    # --- FIX MULTIINDEX ---
    if isinstance(df.columns, pd.MultiIndex):
        # keep only price level (drop ticker level)
        df.columns = df.columns.get_level_values(0)

    df.columns = [c.lower() for c in df.columns]

    return df

df = load_ohlcv(TICKER, START, END)
df.head()

,close,high,low,open,volume
Date,,,,,
2018-01-01,13657.200195,14112.200195,13154.700195,14112.200195,10291200000
2018-01-02,14982.099609,15444.599609,13163.599609,13625.000000,16846600192
2018-01-03,15201.000000,15572.799805,14844.500000,14978.200195,16871900160
2018-01-04,15599.200195,15739.700195,14522.200195,15270.700195,21783199744
2018-01-05,17429.500000,17705.199219,15202.799805,15477.200195,23840899072


# Feature engineering + toy forecasting features

In [4]:
def add_features_and_forecast(df, ewma_span=20, vol_window=20):
    df = df.copy()
    df["log_close"] = np.log(df["close"])
    df["r"] = df["log_close"].diff()

    # Forecast signal (toy): EWMA mean of returns
    df["mu_hat"] = df["r"].ewm(span=ewma_span, adjust=False).mean()

    # Risk estimate: rolling volatility
    df["sigma_hat"] = df["r"].rolling(vol_window).std()

    # Lag features
    df["r_lag1"] = df["r"].shift(1)

    # Momentum
    df["mom_5"] = df["r"].rolling(5).mean()
    df["mom_20"] = df["r"].rolling(20).mean()

    # Volatility regime
    df["vol_ratio"] = df["r"].rolling(10).std() / df["r"].rolling(50).std()

    df = df.dropna()
    return df

df_feat = add_features_and_forecast(df)
df_feat.head()

,close,high,low,open,volume,log_close,r,mu_hat,sigma_hat,r_lag1,mom_5,mom_20,vol_ratio
Date,,,,,,,,,,,,,
2018-02-20,11403.700195,11958.500000,11231.799805,11231.799805,9926540288,9.341693,0.015768,0.015358,0.078997,0.061874,0.022970,0.005474,0.860643
2018-02-21,10690.400391,11418.500000,10479.099609,11372.200195,9405339648,9.277101,-0.064592,0.007744,0.076228,0.015768,0.008728,0.007668,0.884584
2018-02-22,10005.000000,11039.099609,9939.089844,10660.400391,8040079872,9.210840,-0.066261,0.000696,0.077380,-0.064592,-0.021001,0.006242,0.902217
2018-02-23,10301.099609,10487.299805,9734.559570,9937.070312,7739500032,9.240006,0.029166,0.003407,0.077209,-0.066261,-0.004809,0.005789,0.870363
2018-02-24,9813.070312,10597.200195,9546.969727,10287.700195,6917929984,9.191470,-0.048535,-0.001540,0.074067,0.029166,-0.026891,0.008512,0.836170


# Train/Test split (time-based)

In [5]:
n = len(df_feat)
split = int(TRAIN_FRAC * n)

df_train = df_feat.iloc[:split].reset_index(drop=True)
df_test  = df_feat.iloc[split:].reset_index(drop=True)

print(len(df_train), len(df_test))

2353 589


# Trading Environment (target position action)

In [6]:
class TradingEnv(gym.Env):

    metadata = {"render_modes": []}

    def __init__(
            self,
            df,
            fee=0.0005,
            kappa=0.1,
            slippage_coef=0.0,
            smoothing_alpha=1.0,
            max_leverage=1.0,
            reward_scale=1.0,
            include_turnover=False,
            initial_equity=100000.0,
            forecast_probs=None,
    ):
        super().__init__()

        self.df = df.reset_index(drop=True)

        self.fee = float(fee)
        self.kappa = float(kappa)
        self.slippage_coef = float(slippage_coef)

        self.smoothing_alpha = float(smoothing_alpha)
        self.max_leverage = float(max_leverage)

        self.reward_scale = float(reward_scale)

        self.include_turnover = bool(include_turnover)

        self.initial_equity = float(initial_equity)

        self.forecast_probs = forecast_probs
        self.include_forecast = forecast_probs is not None

        self.action_space = spaces.Box(
            low=-self.max_leverage,
            high=self.max_leverage,
            shape=(1,),
            dtype=np.float32
        )

        self.feature_cols = ["r", "r_lag1", "mu_hat", "sigma_hat", "mom_5",	"mom_20", "vol_ratio"]

        portfolio_dim = 3

        if self.include_turnover:
            portfolio_dim += 1

        n_lstm_features = 1 if self.include_forecast else 0

        obs_dim = len(self.feature_cols) + portfolio_dim + n_lstm_features

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(obs_dim,),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.t = 1
        self.pos = 0.0
        self.target_pos = 0.0
        self.prev_turnover = 0.0

        self.equity = self.initial_equity
        self.peak = self.initial_equity

        return self._get_obs(), {}

    def _get_obs(self):

        x = self.df.loc[self.t, self.feature_cols].values.astype(np.float32)

        equity_norm = np.float32(self.equity / self.initial_equity)
        drawdown = np.float32((self.peak - self.equity) / (self.peak + 1e-8))

        portfolio_features = [self.pos, equity_norm, drawdown]

        if self.include_turnover:
            portfolio_features.append(self.prev_turnover)

        obs = np.concatenate(
            [x, np.array(portfolio_features, dtype=np.float32)]
        )

        if self.include_forecast and self.t < len(self.forecast_probs):

            lstm_signal = float(self.forecast_probs[self.t] * 2 - 1)
            obs = np.concatenate([obs, [lstm_signal]])

        elif self.include_forecast:

            obs = np.concatenate([obs, [0.0]])

        return obs

    def step(self, action):

        raw_target = float(np.clip(action[0], -self.max_leverage, self.max_leverage))

        new_pos = (1.0 - self.smoothing_alpha) * self.pos + self.smoothing_alpha * raw_target
        new_pos = float(np.clip(new_pos, -self.max_leverage, self.max_leverage))

        r_t = float(self.df.loc[self.t, "r"])
        sigma_t = float(self.df.loc[self.t, "sigma_hat"])

        if not np.isfinite(sigma_t):
            sigma_t = 0.0

        pnl = self.pos * r_t

        turnover = abs(new_pos - self.pos)

        cost = self.fee * turnover

        slippage = self.slippage_coef * turnover * (1.0 + sigma_t)

        risk_pen = self.kappa * (self.pos ** 2) * sigma_t

        true_reward = pnl - cost - slippage
        reward = true_reward - risk_pen
        reward *= self.reward_scale

        self.target_pos = raw_target
        self.prev_turnover = turnover
        self.pos = new_pos

        self.equity *= float(np.exp(true_reward))
        self.peak = max(self.peak, self.equity)

        self.t += 1

        terminated = (self.t >= len(self.df) - 1)
        truncated = False

        info = {
            "pnl": pnl,
            "cost": cost,
            "slippage": slippage,
            "risk_pen": risk_pen,
            "turnover": turnover,
            "position": self.pos,
            "target_position": self.target_pos,
            "equity": self.equity,
            "drawdown": (self.peak - self.equity) / (self.peak + 1e-8),
            "cumulative_return": (self.equity - self.initial_equity) / self.initial_equity,
        }

        return self._get_obs(), float(reward), terminated, truncated, info

# Vectorized env (train)

In [7]:
def make_env(df):
    def thunk():
        return TradingEnv(df, fee=FEE, kappa=KAPPA,
        slippage_coef=0.001,
        smoothing_alpha=0.3, # was 0.5
        max_leverage=1.0,
        reward_scale=1.0,
        include_turnover=True)
    return thunk

env = gym.vector.SyncVectorEnv([make_env(df_train) for _ in range(num_envs)])
obs_dim = env.single_observation_space.shape[0]
act_dim = env.single_action_space.shape[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("obs_dim:", obs_dim, "act_dim:", act_dim, "device:", device)

obs_dim: 11 act_dim: 1 device: cpu


# PPO model (Gaussian policy + tanh squash + corrected logprob)

In [8]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh()
        )
        self.mu = nn.Linear(64, act_dim)
        self.log_std = nn.Parameter(torch.ones(act_dim) * -1.0)  # good start
        self.v = nn.Linear(64, 1)

    def forward(self, obs):
        x = self.net(obs)
        mu = self.mu(x)
        std = torch.exp(self.log_std)
        dist = Normal(mu, std)
        value = self.v(x).squeeze(-1)
        return dist, value

def squash(u):
    return torch.tanh(u)  # maps to [-1,1]
# main formula:
# a = f(u)
# log p(a) = log p(u) - log |det(Jacobian)|
# log p(a) ist die gesuchte policy log pi(a|a)

# we have f = tanh
# a = tanh(u)
# da/du = 1 - tanh(u)^2
# da/du = 1 - a²
# we need: log |det(Jacobian)|
# we get: log |det(Jacobian)| = log(1 - tanh(u)^2)
# in code: log_det = torch.log(1.0 - torch.tanh(u).pow(2) + eps).sum(-1)

def logprob_squashed(dist, u):
    # log p(u)
    logp_u = dist.log_prob(u).sum(-1)
    # change-of-variables for tanh
    eps = 1e-6
    log_det = torch.log(1.0 - torch.tanh(u).pow(2) + eps).sum(-1)
    return logp_u - log_det

# GAE

In [9]:
def compute_gae(rewards, dones, values, last_value, gamma=0.99, lam=0.95):
    """
    rewards: [T, N]
    dones:   [T, N] (1.0 means terminal boundary for bootstrap mask)
    values:  [T, N]
    last_value: [N]
    """
    T, N = rewards.shape
    adv = torch.zeros(T, N, device=values.device)
    gae = torch.zeros(N, device=values.device)

    for t in reversed(range(T)):
        not_done = 1.0 - dones[t]
        next_value = last_value if t == T - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_value * not_done - values[t]
        gae = delta + gamma * lam * not_done * gae
        adv[t] = gae

    returns = adv + values
    return returns, adv

# PPO training loop

In [10]:
model = ActorCritic(obs_dim, act_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

obs, _ = env.reset(seed=SEED)
obs = torch.as_tensor(obs, dtype=torch.float32, device=device)

# Episode-level tracking
ep_returns = np.zeros(num_envs, dtype=np.float32)
ep_turnover = np.zeros(num_envs, dtype=np.float32)

ep_history = []
turnover_history = []

for update in range(total_updates):
    # Rollout buffers
    obs_buf  = torch.zeros(n_steps, num_envs, obs_dim, device=device)
    u_buf    = torch.zeros(n_steps, num_envs, act_dim, device=device)
    logp_buf = torch.zeros(n_steps, num_envs, device=device)
    rew_buf  = torch.zeros(n_steps, num_envs, device=device)
    done_buf = torch.zeros(n_steps, num_envs, device=device)
    val_buf  = torch.zeros(n_steps, num_envs, device=device)

    # Optional diagnostics per rollout
    # Optional diagnostics per rollout
    pnl_roll = []
    cost_roll = []
    slip_roll = []
    risk_roll = []
    reward_roll = []
    true_reward_roll = []
    equity_roll = []

    for t in range(n_steps):
        obs_buf[t] = obs

        # Collect rollout data WITHOUT gradients
        with torch.no_grad():
            dist, value = model(obs)
            u = dist.sample()
            a = squash(u)
            logp = logprob_squashed(dist, u)

        u_buf[t] = u
        logp_buf[t] = logp.detach()
        val_buf[t] = value.detach()

        next_obs, reward, terminated, truncated, infos = env.step(a.detach().cpu().numpy())
        done_env = np.logical_or(terminated, truncated)

        # For GAE/bootstrap:
        # if you reset on done_env, then bootstrap should also stop on done_env
        done_boot = done_env

        rew_buf[t] = torch.as_tensor(reward, dtype=torch.float32, device=device)
        done_buf[t] = torch.as_tensor(done_boot, dtype=torch.float32, device=device)

        # ---- diagnostics from info dict ----
        # In vector env, infos is usually a dict of arrays
        # ---- diagnostics from info dict ----
        # In vector env, infos is usually a dict of arrays
        if isinstance(infos, dict):
            if "pnl" in infos:
                pnl_roll.append(np.mean(infos["pnl"]))
            if "cost" in infos:
                cost_roll.append(np.mean(infos["cost"]))
            if "slippage" in infos:
                slip_roll.append(np.mean(infos["slippage"]))
            if "risk_pen" in infos:
                risk_roll.append(np.mean(infos["risk_pen"]))
            if "reward" in infos:
                reward_roll.append(np.mean(infos["reward"]))
            if "true_reward" in infos:
                true_reward_roll.append(np.mean(infos["true_reward"]))
            if "equity" in infos:
                equity_roll.append(np.mean(infos["equity"]))
            if "turnover" in infos:
                ep_turnover += infos["turnover"]

        # ---- episode return tracking ----
        ep_returns += reward

        if done_env.any():
            finished = np.where(done_env)[0]

            ep_history.extend(ep_returns[finished].tolist())
            turnover_history.extend(ep_turnover[finished].tolist())

            ep_returns[finished] = 0.0
            ep_turnover[finished] = 0.0

            # IMPORTANT: reset only finished envs
            next_obs, _ = env.reset(options={"reset_mask": done_env})

        obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)

    # Bootstrap last value
    with torch.no_grad():
        _, last_value = model(obs)

    returns, adv = compute_gae(
        rew_buf,
        done_buf,
        val_buf,
        last_value,
        gamma=gamma,
        lam=gae_lambda
    )

    # Flatten rollout
    B = n_steps * num_envs
    obs_batch  = obs_buf.reshape(B, obs_dim).detach()
    u_batch    = u_buf.reshape(B, act_dim).detach()
    old_logp   = logp_buf.reshape(B).detach()
    old_value  = val_buf.reshape(B).detach()
    ret_batch  = returns.reshape(B).detach()
    adv_batch  = adv.reshape(B).detach()

    # Normalize advantages
    adv_batch = (adv_batch - adv_batch.mean()) / (adv_batch.std() + 1e-8)

    idx = torch.arange(B, device=device)
    stop = False
    last_kl = 0.0

    for _ in range(ppo_epochs):
        perm = idx[torch.randperm(B)]
        for start in range(0, B, minibatch_size):
            mb = perm[start:start + minibatch_size]

            dist, value = model(obs_batch[mb])
            logp = logprob_squashed(dist, u_batch[mb])
            entropy = dist.entropy().sum(-1)

            # Approximate KL
            approx_kl = (old_logp[mb] - logp).mean().detach()
            last_kl = float(approx_kl.item())

            if last_kl > target_kl:
                stop = True
                break

            ratio = torch.exp(logp - old_logp[mb])

            # PPO clipped policy objective
            unclipped = ratio * adv_batch[mb]
            clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv_batch[mb]
            policy_loss = -torch.min(unclipped, clipped).mean()

            # Value loss (simple version; can later replace with clipped value loss)
            value_loss = (ret_batch[mb] - value).pow(2).mean()

            # Entropy bonus
            entropy_loss = -entropy.mean()

            loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        if stop:
            break

        # Keep std in sane range
        with torch.no_grad():
             model.log_std.clamp_(-1.5, -0.5)
            # model.log_std.clamp_(-2.0, -0.5)

    # ---- logging ----
    if update % 100 == 0:
        mean_100 = np.mean(ep_history[-100:]) if len(ep_history) >= 100 else np.nan
        mean_turnover = np.mean(turnover_history[-100:]) if len(turnover_history) >= 100 else np.nan

        mean_pnl = np.mean(pnl_roll) if len(pnl_roll) > 0 else np.nan
        mean_cost = np.mean(cost_roll) if len(cost_roll) > 0 else np.nan
        mean_slip = np.mean(slip_roll) if len(slip_roll) > 0 else np.nan
        mean_risk = np.mean(risk_roll) if len(risk_roll) > 0 else np.nan

        mean_reward = np.mean(reward_roll) if len(reward_roll) > 0 else np.nan
        mean_true_reward = np.mean(true_reward_roll) if len(true_reward_roll) > 0 else np.nan
        mean_equity = np.mean(equity_roll) if len(equity_roll) > 0 else np.nan

        print(
            f"Update {update:4d} | "
            f"mean_return(last100) {mean_100:8.3f} | "
            f"turnover(last100) {mean_turnover:8.3f} | "
            f"KL {last_kl:8.4f} | "
            f"log_std {model.log_std.data.cpu().numpy()} | "
            f"equity {mean_equity:8.4f}"
        )
        print(
            f"   rollout diagnostics -> "
            f"reward: {mean_reward:.5f}, true_reward: {mean_true_reward:.5f}, "
            f"pnl: {mean_pnl:.5f}, cost: {mean_cost:.5f}, "
            f"slippage: {mean_slip:.5f}, risk_pen: {mean_risk:.5f}"
        )

Update    0 | mean_return(last100)      nan | turnover(last100)      nan | KL   0.0136 | log_std [-1.0011432] | equity   0.9856
   rollout diagnostics -> reward: -0.00030, true_reward: -0.00030, pnl: -0.00017, cost: 0.00004, slippage: 0.00009, risk_pen: 0.00000


KeyboardInterrupt: 

# Evaluation on test set (single env, deterministic actions)

In [ ]:
def eval_policy(model, df_eval, episodes=5):
    env_eval = TradingEnv(df_eval, fee=FEE, kappa=KAPPA,
        slippage_coef=0.001,
        smoothing_alpha=0.5,
        max_leverage=1.0,
        reward_scale=1.0,
        include_turnover=True)

    returns =[]

    for _ in range(episodes):
        obs, _ = env_eval.reset()
        done = False
        ep_ret = 0.0

        while not done:
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                dist, _ = model(obs_t)
                # Deterministic: use mean action (mu), then squash
                u = dist.mean
                a = squash(u).cpu().numpy()[0]

            obs, reward, terminated, truncated, _ = env_eval.step(a)
            done = terminated or truncated
            ep_ret += reward

        returns.append(ep_ret)

    return float(np.mean(returns))

test_score = eval_policy(model, df_test, episodes=10)
print("EVAL mean episode reward:", test_score)

# Equity curve plot (test set, one run)

In [ ]:
def run_equity_curve(model, df_eval):
    env_eval = TradingEnv(df_eval, fee=FEE, kappa=KAPPA,
        slippage_coef=0.001,
        smoothing_alpha=0.5,
        max_leverage=1.0,
        reward_scale=1.0,
        include_turnover=True)

    obs, _ = env_eval.reset()
    done = False

    equity = [env_eval.equity]
    pos_hist = [env_eval.pos]

    while not done:
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            dist, _ = model(obs_t)
            u = dist.mean
            a = squash(u).cpu().numpy()[0]

        obs, reward, terminated, truncated, _ = env_eval.step(a)
        done = terminated or truncated
        equity.append(env_eval.equity)
        pos_hist.append(env_eval.pos)

    return np.array(equity), np.array(pos_hist)

equity, pos_hist = run_equity_curve(model, df_test)

plt.figure()
plt.plot(equity)
plt.title("Equity Curve (Test)")
plt.xlabel("Step")
plt.ylabel("Equity")
plt.show()

plt.figure()
plt.plot(pos_hist)
plt.title("Position (Test)")
plt.xlabel("Step")
plt.ylabel("Position [-1, 1]")
plt.show()

In [ ]:
# env = TradingEnv_2(df_train, fee=0.0005, kappa=0.1)
# env = TradingEnv_2(df_train, fee=0.0005, kappa=0.1, slippage_coef=0.001)
# env = TradingEnv_2(
#    df_train,
#    fee=0.0005,
#    kappa=0.1,
#    slippage_coef=0.001,
#    smoothing_alpha=0.3,
#   include_turnover=True,
#)

In [ ]:
# extendself.feature_cols = ["r", "r_lag1", "mu_hat", "sigma_hat"]
#  self.feature_cols = ["r", "r_lag1", "mu_hat", "sigma_hat", "mom_5", "mom_20"]....

**Version 1 — easier reward**

kappa = 0.01

slippage_coef = 0.0

**Version 2 — moderate realism**

kappa = 0.02

slippage_coef = 0.001

**Version 3 — more realistic**

kappa = 0.05

slippage_coef = 0.001

smoothing on



At which level of realism does PPO still learn?